<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div>
</div>
<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;"><h1 style="margin:0;"><b>Feature Detection and Object Tracking — Algorithm</b></h1></div>


## Purpose and Reading Convention

This notebook explains how to implement and review this lab inside its existing `labs/` directory. The executable code remains in [main.ipynb](main.ipynb); no repository split or Python package extraction is required.

Each numbered step gives **Purpose**, **Inputs**, **Procedure**, **Outputs**, **Validation** and **On failure**. Follow dependencies in order and reuse the same intermediate arrays for metrics and figures.

**Validation convention:** these are review contracts. Some are enforced by existing assertions or exceptions; others require inspecting intermediate results. A documentation update does not add runtime guards. The *Implementation limits* section identifies important gaps so the algorithm does not claim unimplemented protections.

Use parameter values and numeric tolerances actually declared in `main.ipynb`. Do not replace them with unsupported universal thresholds. A failed implementation check stops the run; a diagnostic limitation must be reported rather than silently treated as a pass.

## Inputs and Final Outputs

| Item | Description |
| --- | --- |
| **Inputs** | Input video, initial object bounding box, ORB and RANSAC parameters |
| **Final outputs** | Per-frame tracking records, transformed object boxes, match/inlier metrics, five diagnostic figures |
| **Implementation target** | `main.ipynb` |

## 0. Start

**Purpose**

Establish the scope and a clean execution order.

**Inputs**

The problem statement, theory, requirements and main notebook in this lab.

**Procedure**

1. Select the project environment and open the tracking lab in VS Code.
2. Use this notebook as the exact coding order for `main.ipynb`.
3. Keep the first frame fixed as the reference throughout the sequence.

**Outputs**

Selected environment and a top-to-bottom execution plan.

**Validation**

Review the local assumptions before using cached notebook outputs.

**On failure**

Resolve missing prerequisites first; do not infer a successful run from stored outputs.

## 1. Create Paths and Discover the Video

**Purpose**

Select a reproducible video source.

**Inputs**

Supported videos under the lab data directory.

**Procedure**

1. Import `Path`, OpenCV, NumPy, and Matplotlib.
2. Create repository-local data and output paths.
3. Discover supported videos recursively and sort them deterministically.
4. Select the intended video by index and fail if no valid video exists.

**Outputs**

Sorted candidates; VIDEO_PATH; OUTPUT_DIR.

**Validation**

At least one candidate; VIDEO_INDEX identifies the intended sequence.

**On failure**

Stop if no supported file exists or selection is out of range.

## 2. Define the Initial Object Region and Parameters

**Purpose**

Specify the object geometry in reference-frame coordinates.

**Inputs**

ROW=24, COL=46, HEIGHT=170, WIDTH=160.

**Procedure**

1. Define row, column, height, and width of the reference object.
2. Convert them into a four-corner float32 bounding box shaped `(4,1,2)`.
3. Define maximum ORB features, RANSAC threshold, minimum matches, and minimum inliers.
4. Validate the bounding-box shape before opening the sequence.

**Outputs**

float32 initial_bounding_box shaped (4,1,2); ORB/RANSAC settings.

**Validation**

1000 features; threshold 5 pixels; minimum 4 matches and 4 inliers. Review ROI bounds against the decoded frame.

**On failure**

Correct an invalid ROI before extracting object features.

## 3. Initialize Feature Detection and Matching

**Purpose**

Use a distance suited to binary ORB descriptors.

**Inputs**

MAX_FEATURES and matching configuration.

**Procedure**

1. Create ORB with the configured feature limit.
2. Create a brute-force matcher with Hamming distance for binary ORB descriptors.
3. Enable `crossCheck=True` to retain reciprocal matches.
4. Keep the detector and matcher fixed for all frames.

**Outputs**

ORB detector; BFMatcher with Hamming and crossCheck.

**Validation**

Reciprocal matches compare binary descriptors; detector/matcher remain fixed.

**On failure**

Do not substitute Euclidean descriptor distance without changing the descriptor method.

## 4. Open and Validate the Reference Frame

**Purpose**

Establish a fixed reference view.

**Inputs**

VIDEO_PATH.

**Procedure**

1. Open the selected video with `VideoCapture` and verify it opened.
2. Read reported frame count when available.
3. Read the first frame and stop if decoding fails.
4. Convert the reference frame to grayscale.

**Outputs**

First decoded BGR frame, grayscale frame, reported frame count.

**Validation**

Capture opens and first frame decodes; metadata may be unavailable.

**On failure**

Release capture and stop on reference-frame failure.

## 5. Extract Reference Features Inside the Object

**Purpose**

Restrict reference features to the object.

**Inputs**

Reference grayscale frame and ROI mask.

**Procedure**

1. Create a zero mask matching the reference-frame size.
2. Fill only the initial object rectangle with 255.
3. Run ORB `detectAndCompute` using that mask.
4. Require at least four descriptors and descriptor width 32.
5. Preserve these reference keypoints/descriptors for the full sequence.

**Outputs**

Reference keypoints and descriptors shaped (N,32).

**Validation**

At least 4 descriptors; nonempty ROI within the image.

**On failure**

Stop on insufficient reference features; changing the ROI requires rerunning extraction.

## 6. Save the Reference Diagnostic

**Purpose**

Verify the reference region visually.

**Inputs**

Reference frame, box and keypoints.

**Procedure**

1. Draw the initial object boundary on a reference-frame copy.
2. Draw the reference ORB keypoints.
3. Convert BGR to RGB for Matplotlib.
4. Save `reference_orb_keypoints.png`.

**Outputs**

reference_orb_keypoints.png.

**Validation**

Draw on a copy; convert BGR to RGB; keypoints correspond to stored descriptors.

**On failure**

Correct ROI or channel display errors before tracking.

## 7. Implement Frame Matching and Homography Estimation

**Purpose**

Estimate the reference-to-current mapping robustly.

**Inputs**

Current grayscale frame and fixed reference features.

**Procedure**

1. Detect ORB keypoints/descriptors in the current grayscale frame.
2. If descriptors are missing or too few, return a structured failure record.
3. Match reference descriptors to current descriptors and sort by distance.
4. If fewer than four matches remain, return failure.
5. Build paired reference/current point arrays.
6. Estimate the homography with RANSAC.
7. Reject missing, non-finite, or wrongly shaped homographies.
8. Count inliers and reject results below the minimum.
9. Return success state, homography, inlier mask, keypoints, matches, and counts.

**Outputs**

Structured success/failure record, H, matches and inlier mask.

**Validation**

At least 4 matches/inliers; finite 3 by 3 H; inlier count cannot exceed match count.

**On failure**

Return the existing failure reason for missing features, matches, H or inliers; do not invent a transform.

## 8. Track Through the Complete Video

**Purpose**

Record both successful and failed frame estimates.

**Inputs**

Video frames after the reference; matching helper.

**Procedure**

1. Start from the frame immediately after the reference frame.
2. For each frame: convert to grayscale and call the matching helper.
3. On success, transform the four reference-box corners through the homography.
4. Store frame index, homography, transformed box, match count, and inlier count.
5. On failure, store frame index and failure reason instead of inventing a box.
6. Continue until the video ends, then release the capture.

**Outputs**

Frame-indexed tracking records with transformed boxes on success.

**Validation**

Each successful box is finite (4,1,2); failed frames remain represented.

**On failure**

Continue after a recorded tracking failure; release capture after decoding ends.

## 9. Compute Sequence Metrics

**Purpose**

Quantify success over the processed sequence.

**Inputs**

All tracking records, including failures.

**Procedure**

1. Extract successful tracking records.
2. Stop if no valid homography was produced.
3. Create arrays of frame indices, matches, inliers, and inlier ratios.
4. Compute tracking success rate and summary statistics.

**Outputs**

Success rate; per-frame match/inlier/ratio arrays.

**Validation**

Success denominator is all processed frames; ratios stay in [0,1]; at least one valid result.

**On failure**

Stop if every frame fails; do not compute success rate on successful frames alone.

## 10. Implement Random-Access Frame Reading

**Purpose**

Retrieve frames for diagnostics without altering tracking records.

**Inputs**

Video path and recorded frame index.

**Procedure**

1. Create a helper that reopens the video and seeks to a requested frame index.
2. Read exactly one frame, release the capture, and fail if decoding fails.
3. Use this helper only for visualization; do not rerun tracking state.

**Outputs**

One decoded frame.

**Validation**

Requested index corresponds to the stored result; capture is released.

**On failure**

Raise on failed random-access decoding rather than using another frame.

## 11. Visualize Representative Tracking Frames

**Purpose**

Show tracking quality across time.

**Inputs**

Up to six successful records spread over the sequence.

**Procedure**

1. Choose up to six successful results uniformly across the sequence.
2. Read each corresponding frame.
3. Draw the stored transformed bounding box and annotate frame/inlier information.
4. Save the representative tracking grid.

**Outputs**

representative_tracking_frames.png.

**Validation**

Draw stored boxes on the corresponding frames; label frame and inlier counts.

**On failure**

Report unavailable frames; successful-frame sampling alone does not represent failures.

## 12. Visualize RANSAC Inlier Matches

**Purpose**

Show which descriptor matches support a homography.

**Inputs**

A successful middle frame and fixed reference.

**Procedure**

1. Choose a successful frame near the middle of the sequence.
2. Recompute descriptor matching for that frame to obtain match objects.
3. Use the RANSAC mask so only geometrically consistent matches are highlighted.
4. Draw reference/current boxes and save the inlier-match diagnostic.

**Outputs**

ransac_inlier_matches.png.

**Validation**

Current implementation reruns matching for this diagnostic and draws its inliers.

**On failure**

If the rerun fails, report it; stochastic RANSAC reruns need not reproduce the stored inlier set exactly.

## 13. Plot Match and Inlier Evolution

**Purpose**

Expose weak frames through temporal diagnostics.

**Inputs**

Recorded frame indices, matches, inliers and ratios.

**Procedure**

1. Plot descriptor-match and RANSAC-inlier counts against frame index.
2. Plot inlier ratio against frame index separately.
3. Use the curves to expose weak frames even when a box exists.
4. Save both figures.

**Outputs**

Two temporal plots.

**Validation**

Preserve sequence indices and failure records; verify 0<=inliers<=matches.

**On failure**

Investigate gaps or impossible counts rather than smoothing them away.

## 14. Run Final Validation

**Purpose**

Check geometric records and required outputs.

**Inputs**

Reference descriptors, sequence records, video metadata and five files.

**Procedure**

1. Verify reference descriptors exist, have enough rows, and width 32.
2. Verify processed-frame count is consistent with video metadata when available.
3. For every success verify a finite 3×3 homography and finite `(4,1,2)` box.
4. Verify match/inlier counts and ratios are valid.
5. Verify all declared output files exist before printing the pass message.

**Outputs**

Final validation outcome.

**Validation**

N by 32 descriptors; finite 3 by 3 H and (4,1,2) boxes on success; valid counts and ratios.

**On failure**

Stop on failed invariants or missing figures; inspect metadata disagreements as decoding/container limitations.

## Complete Formal Pseudocode

**Input:** Video; reference-frame ROI (row=24,column=46,height=170,width=160); ORB/RANSAC settings.

**Output:** Per-frame success or failure records; homography (3,3) and box (4,1,2) on success; match/inlier statistics; five figures.

**Data contract:** A homography maps fixed first-frame coordinates to the current frame. Descriptors have shape (N,32), uint8. Box coordinates are (x,y), while image indexing is [y,x].

The following numbered blocks form **one complete execution algorithm**. They include every lab-specific helper body, every numerical expression, every loop/branch and the existing validation/diagnostic steps. Inputs, literal parameters and output paths are assigned explicitly in the setup blocks; intermediate variables are produced before use in execution order. Each procedure's `RETURN` identifies its result. Top-level results are named by the output contract above.

**Scope:** this is a formal transcription of the current executable workflow, including its explicitly documented limitations. Additional physical-validity requirements in the surrounding review text are not silently presented as implemented guards. Error branches below specify the actual termination/rejection behaviour; the implementation-limit section identifies missing protections.

### Pseudocode notation and execution rules

- `x ← expression` assigns a value; `=` compares values. `A · B` is matrix multiplication; `*`, `/` and `^` are elementwise arithmetic for arrays and scalar arithmetic for numbers. Explicit dtype conversions are retained because clipping and overflow are part of these labs.
- Arrays are **zero-indexed**. `a:b:s` is an end-exclusive slice with step `s`; omitted limits use the array boundary. `RANGE(n)` produces `0,…,n−1`; `RANGE(a,b,s)` excludes `b`. Tuple assignment unpacks corresponding values. `TRANSPOSE(A)` interchanges axes as the array transpose operation does.
- `LIST[expression FOR x IN sequence WHERE condition]` constructs a list in iteration order. `STREAM[...]` is lazy; `SET{...}` deduplicates; `MAP{key ↦ value FOR ...}` creates a mapping. `EXPAND` inserts a sequence or mapping's members. `FUNCTION(parameters) ↦ expression` defines a callable expression.
- `IF`, `FOR`, `WHILE`, `RETURN`, `BREAK` and `CONTINUE` have their usual structured meanings and explicit closing statements. `REQUIRE condition OTHERWISE ...` stops on failure. `RAISE Error(message)` propagates that error; it is not a successful return. A logged frame/view rejection followed by `CONTINUE` is deliberately different from stopping the experiment.
- `AND`/`OR` short-circuit scalar conditions. `AND_EACH`/`OR_EACH` are elementwise operations for boolean masks and bitwise operations for integer flags. `COMPLEMENT` complements boolean masks or integer bits according to dtype. `NULL`, `TRUE`, `FALSE`, `ARRAY.nan` and `ARRAY.inf` retain their distinct meanings; `IS` tests identity.
- `PROCEDURE` defines the complete body shown below; indentation determines scope. Default arguments refer to the environment at definition time. `CLASS` defines stateful objects, `self` is the current instance, `__init__` initializes it, and `STATIC` methods need no instance. `NO_GRADIENT` disables automatic differentiation throughout the procedure. Each block continues the shared state of the preceding block; run them **once, top to bottom**, starting with a fresh environment.
- `CAST`, `RESHAPE`, `FLATTEN`, `COPY`, `MEAN`, `SUM`, `MIN`, `MAX`, `STD`, `APPEND`, `EXTEND`, `ITEMS`, `KEYS` and `VALUES` are standard array/container operations. `CAST` defaults to a new array; `FLATTEN` follows row-major order; axis arguments specify reductions. `TO_DEVICE`, `TO_CPU`, `TO_ARRAY`, `SCALAR_VALUE`, `ELEMENT_COUNT` and `TO_LIST` perform the named tensor/container conversions. `SET_MODE(model,TRAINING/EVALUATION)` switches neural-network behaviour.
- `LENGTH`, `SORT`, `ZIP`, `ENUMERATE`, `ABS`, `ALL`, `ANY`, `INTEGER`, `REAL`, `STRING`, `BOOLEAN`, `TUPLE`, `LIST`, `SET`, `MAP`, `GET_ATTRIBUTE`, `IS_INSTANCE` and `ROUND` are standard built-ins with the indicated roles. `super().__init__()` initializes the declared base class. `FORMAT` uses numbered placeholders and the explicitly retained numeric format specifications; `EMIT` writes a diagnostic. These are presentation operations, not hidden numerical algorithms.

### Standard-library primitive boundary

Calls under the following namespaces are explicitly delegated to standard libraries. Their named parameters and constants are shown at each call; all **lab-specific** procedures are defined in full below. Constructors, file objects, axes, video captures, optimizers and tensors retain the methods of their declared library. This fixes interpolation, boundary, dtype and random-state semantics without pretending to reimplement SVD, FFT, ORB, automatic differentiation or image codecs.

| Pseudocode namespace | Library / primitive family |
| --- | --- |
| `ARRAY` | NumPy arrays, broadcasting, linear algebra/SVD, FFT, statistics, RNG and window views |
| `CV` | OpenCV image conversion, chessboard detection/subpixel refinement, ORB/Hamming matching, RANSAC, morphology and watershed |
| `NDIMAGE`, `INTERPOLATE_COORDINATES` | SciPy ndimage filters, morphology and coordinate interpolation |
| `TENSOR`, `LAYERS` | PyTorch tensors, devices, differentiation, optimizers, layers and cross-entropy |
| `DATASET_INTERFACE`, `BATCH_LOADER` | PyTorch Dataset protocol and DataLoader batching, with the explicit local collation method |
| `READ_MNIST_IDX` | python-mnist IDX reader; `load_training` and `load_testing` return images and labels |
| `IMAGE` | Pillow image loading, mode conversion, resizing and encoding |
| `PATH` | Pathlib path joining, traversal, existence checks and output-directory creation; `/` joins path components for path operands |
| `PLOT` | Matplotlib figures, axes, drawing and saving; named arrays are the plotted data |
| `MATH`, `PROGRESS` | Standard scalar math and tqdm progress iteration |

`DATASET_INTERFACE` and `BATCH_LOADER` are standard abstractions; `MNISTDataset` and `MNISTClassifier`, where used, are fully defined below. Visualization calls preserve the figures' data, labels and output filenames. They can be translated to another renderer without changing the numerical procedure.

### Algorithm 1 — Setup — Environment and Configuration

```text
CALL ARRAY.set_printoptions(precision = 4, suppress = TRUE)
OUTPUT_DIR ← PATH("../outputs/figures")
CALL OUTPUT_DIR.mkdir(parents = TRUE, exist_ok = TRUE)
CALL EMIT(FORMAT("OpenCV version: {0}" , CV.__version__))
```

### Algorithm 2 — 1. Validate the Input Video and Output Paths

```text
DATA_DIR ← PATH("../data")
VIDEO_EXTENSIONS ← {".mp4", ".avi", ".mov", ".mkv"}
discovered_video_paths ← SORT(STREAM[path FOR path IN DATA_DIR.rglob("*") WHERE (path.is_file() AND (path.suffix.lower() IN VIDEO_EXTENSIONS))])
IF (NOT discovered_video_paths) THEN
    RAISE FileNotFoundError(FORMAT("No supported video found in: {0}" , DATA_DIR))
END IF
VIDEO_INDEX ← 0
VIDEO_PATH ← discovered_video_paths[VIDEO_INDEX]
CALL OUTPUT_DIR.mkdir(parents = TRUE, exist_ok = TRUE)
CALL EMIT(FORMAT("Videos discovered: {0}" , LENGTH(discovered_video_paths)))
CALL EMIT(FORMAT("Selected video   : {0}" , VIDEO_PATH.name))
CALL EMIT(FORMAT("Figures directory: {0}" , OUTPUT_DIR))
```

### Algorithm 3 — 2. Define the Initial Bounding Box and Tracking Parameters

```text
ROW ← 24
COL ← 46
HEIGHT ← 170
WIDTH ← 160
MAX_FEATURES ← 1000
RANSAC_THRESHOLD ← 5.0
MIN_MATCHES ← 4
MIN_INLIERS ← 4
initial_bounding_box ← RESHAPE(ARRAY.float32([[COL, ROW], [(COL + WIDTH), ROW], [(COL + WIDTH),
    (ROW + HEIGHT)], [COL, (ROW + HEIGHT)]]), (−1), 1, 2)
IF (initial_bounding_box.shape ≠ (4, 1, 2)) THEN
    RAISE ValueError(FORMAT("Unexpected bounding-box shape: {0}" , initial_bounding_box.shape))
END IF
CALL EMIT("Initial bounding box corners:")
CALL EMIT(RESHAPE(initial_bounding_box, (−1), 2))
CALL EMIT(FORMAT("ORB features: {0} | RANSAC threshold: {1:.1f} px" , MAX_FEATURES, RANSAC_THRESHOLD))
```

### Algorithm 4 — 3. Initialize ORB and the Hamming-Distance Matcher

```text
orb ← CV.ORB_create(nfeatures = MAX_FEATURES)
matcher ← CV.BFMatcher(CV.NORM_HAMMING, crossCheck = TRUE)
CALL EMIT("ORB detector initialized.")
CALL EMIT("BFMatcher metric: Hamming | crossCheck=True")
```

### Algorithm 5 — 4. Read and Validate the Reference Frame

```text
cap ← CV.VideoCapture(STRING(VIDEO_PATH))
IF (NOT cap.isOpened()) THEN
    RAISE RuntimeError(FORMAT("Could not open video: {0}" , VIDEO_PATH))
END IF
reported_frame_count ← INTEGER(cap.get(CV.CAP_PROP_FRAME_COUNT))
(ret, reference_frame) ← cap.read()
CALL cap.release()
IF ((NOT ret) OR (reference_frame IS NULL)) THEN
    RAISE RuntimeError("Could not read the first video frame.")
END IF
reference_gray_frame ← CV.cvtColor(reference_frame, CV.COLOR_BGR2GRAY)
(frame_height, frame_width) ← reference_gray_frame.shape
IF ((ROW < 0) OR (COL < 0) OR ((ROW + HEIGHT) > frame_height) OR ((COL + WIDTH) > frame_width)) THEN
    RAISE ValueError("The initial bounding box lies outside the reference frame.")
END IF
CALL EMIT(FORMAT("Reference frame shape: {0}" , reference_frame.shape))
CALL EMIT(FORMAT("Reported video frames: {0}" , reported_frame_count))
```

### Algorithm 6 — 5. Detect Reference ORB Features Inside the Object Region

```text
reference_region_mask ← ARRAY.zeros(reference_gray_frame.shape, dtype = ARRAY.uint8)
reference_region_mask[(ROW:(ROW + HEIGHT), COL:(COL + WIDTH))] ← 255
(reference_keypoints, reference_descriptors) ← orb.detectAndCompute(reference_gray_frame,
    reference_region_mask)
IF ((reference_descriptors IS NULL) OR (LENGTH(reference_keypoints) < MIN_MATCHES)) THEN
    RAISE RuntimeError("Not enough ORB features were detected inside the initial bounding box.")
END IF
IF ((reference_descriptors.ndim ≠ 2) OR (reference_descriptors.shape[1] ≠ 32)) THEN
    RAISE ValueError(FORMAT("Unexpected ORB descriptor shape: {0}" , reference_descriptors.shape))
END IF
CALL EMIT(FORMAT("Reference keypoints detected: {0}" , LENGTH(reference_keypoints)))
CALL EMIT(FORMAT("Descriptor shape: {0}" , reference_descriptors.shape))
```

### Algorithm 7 — 6. Visualize the Reference Object and ORB Keypoints

```text
reference_with_bbox ← COPY(reference_frame)
CALL CV.polylines(reference_with_bbox, [ARRAY.int32(initial_bounding_box)], TRUE, (0, 255, 0), 3,
    CV.LINE_AA)
reference_feature_visualization ← CV.drawKeypoints(reference_with_bbox, reference_keypoints, NULL,
    flags = CV.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
reference_feature_visualization_rgb ← CV.cvtColor(reference_feature_visualization, CV.COLOR_BGR2RGB)
(fig, ax) ← PLOT.subplots(figsize = (10, 6))
CALL ax.imshow(reference_feature_visualization_rgb)
CALL ax.set_title("Reference Object and ORB Keypoints")
CALL ax.axis("off")
CALL PLOT.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "reference_orb_keypoints.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
```

### Algorithm 8 — 7. Define Frame Matching and RANSAC Homography Estimation

```text
PROCEDURE match_features_and_estimate_homography(gray_frame, reference_keypoints, reference_descriptors,
    orb_detector, descriptor_matcher, min_matches = MIN_MATCHES, min_inliers = MIN_INLIERS,
    ransac_threshold = RANSAC_THRESHOLD)
    (keypoints, descriptors) ← orb_detector.detectAndCompute(gray_frame, NULL)
    IF ((descriptors IS NULL) OR (LENGTH(keypoints) < min_matches)) THEN
        RETURN {"success" ↦ FALSE, "reason" ↦ "insufficient_current_features", "num_matches" ↦ 0,
            "num_inliers" ↦ 0}
    END IF
    matches ← descriptor_matcher.match(reference_descriptors, descriptors)
    matches ← SORT(matches, key = FUNCTION(match) ↦ match.distance)
    IF (LENGTH(matches) < min_matches) THEN
        RETURN {"success" ↦ FALSE, "reason" ↦ "insufficient_matches", "num_matches" ↦ LENGTH(matches),
            "num_inliers" ↦ 0}
    END IF
    matched_reference_points ← RESHAPE(ARRAY.float32(LIST[reference_keypoints[match.queryIdx].pt FOR match IN matches]),
        (−1), 2)
    matched_current_points ← RESHAPE(ARRAY.float32(LIST[keypoints[match.trainIdx].pt FOR match IN matches]),
        (−1), 2)
    (homography, inlier_mask) ← CV.findHomography(matched_reference_points, matched_current_points,
        CV.RANSAC, ransac_threshold)
    IF ((homography IS NULL) OR (inlier_mask IS NULL) OR (homography.shape ≠ (3,
        3)) OR (NOT ARRAY.all(ARRAY.isfinite(homography)))) THEN
        RETURN {"success" ↦ FALSE, "reason" ↦ "homography_failed", "num_matches" ↦ LENGTH(matches),
            "num_inliers" ↦ 0}
    END IF
    inlier_mask ← CAST(FLATTEN(inlier_mask), BOOLEAN)
    num_inliers ← INTEGER(SUM(inlier_mask))
    IF (num_inliers < min_inliers) THEN
        RETURN {"success" ↦ FALSE, "reason" ↦ "insufficient_inliers", "num_matches" ↦ LENGTH(matches),
            "num_inliers" ↦ num_inliers}
    END IF
    RETURN {"success" ↦ TRUE, "reason" ↦ NULL, "keypoints" ↦ keypoints, "matches" ↦ matches,
        "homography" ↦ homography, "inlier_mask" ↦ inlier_mask, "num_matches" ↦ LENGTH(matches),
        "num_inliers" ↦ num_inliers}
END PROCEDURE
CALL EMIT("Frame matching and homography estimation function defined.")
```

### Algorithm 9 — 8. Track the Object Throughout the Video

```text
cap ← CV.VideoCapture(STRING(VIDEO_PATH))
IF (NOT cap.isOpened()) THEN
    RAISE RuntimeError(FORMAT("Could not open video: {0}" , VIDEO_PATH))
END IF
frame_tracking_results ← []
current_frame_index ← 0
WHILE TRUE DO
    (ret, frame) ← cap.read()
    IF (NOT ret) THEN
        BREAK
    END IF
    IF (current_frame_index = 0) THEN
        current_frame_index ← (current_frame_index + 1)
        CONTINUE
    END IF
    gray_frame ← CV.cvtColor(frame, CV.COLOR_BGR2GRAY)
    result ← match_features_and_estimate_homography(gray_frame = gray_frame,
        reference_keypoints = reference_keypoints, reference_descriptors = reference_descriptors,
        orb_detector = orb, descriptor_matcher = matcher)
    tracking_summary ← {"frame_index" ↦ current_frame_index, "success" ↦ result["success"],
        "reason" ↦ result["reason"], "num_matches" ↦ result["num_matches"],
        "num_inliers" ↦ result["num_inliers"]}
    IF result["success"] THEN
        tracked_bounding_box ← CV.perspectiveTransform(initial_bounding_box, result["homography"])
        IF (NOT ARRAY.all(ARRAY.isfinite(tracked_bounding_box))) THEN
            tracking_summary["success"] ← FALSE
            tracking_summary["reason"] ← "invalid_projected_bbox"
        ELSE
            tracking_summary["homography"] ← result["homography"]
            tracking_summary["current_bbox"] ← tracked_bounding_box
        END IF
    END IF
    CALL APPEND(frame_tracking_results, tracking_summary)
    current_frame_index ← (current_frame_index + 1)
END WHILE
CALL cap.release()
successful_tracking_results ← LIST[result FOR result IN frame_tracking_results WHERE result["success"]]
failed_tracking_results ← LIST[result FOR result IN frame_tracking_results WHERE (NOT result["success"])]
CALL EMIT(FORMAT("Frames processed: {0}" , LENGTH(frame_tracking_results)))
CALL EMIT(FORMAT("Successful tracking frames: {0}" , LENGTH(successful_tracking_results)))
CALL EMIT(FORMAT("Failed tracking frames: {0}" , LENGTH(failed_tracking_results)))
```

### Algorithm 10 — 9. Compute Tracking Summary Metrics

```text
IF (NOT successful_tracking_results) THEN
    RAISE RuntimeError("No frame produced a valid homography. Check the video, ROI, or matching parameters.")
END IF
tracked_frame_indices ← ARRAY.asarray(LIST[result["frame_index"] FOR result IN successful_tracking_results],
    dtype = INTEGER)
feature_match_counts ← ARRAY.asarray(LIST[result["num_matches"] FOR result IN successful_tracking_results],
    dtype = INTEGER)
homography_inlier_counts ← ARRAY.asarray(LIST[result["num_inliers"] FOR result IN successful_tracking_results],
    dtype = INTEGER)
homography_inlier_ratios ← (homography_inlier_counts / feature_match_counts)
success_rate ← (LENGTH(successful_tracking_results) / LENGTH(frame_tracking_results))
CALL EMIT(FORMAT("Tracking success rate: {0:.3f}" , success_rate))
CALL EMIT(FORMAT("Mean matches per successful frame: {0:.2f}" , MEAN(feature_match_counts)))
CALL EMIT(FORMAT("Mean RANSAC inliers per successful frame: {0:.2f}" , MEAN(homography_inlier_counts)))
CALL EMIT(FORMAT("Mean inlier ratio: {0:.3f}" , MEAN(homography_inlier_ratios)))
CALL EMIT(FORMAT("Minimum inlier ratio: {0:.3f}" , MIN(homography_inlier_ratios)))
CALL EMIT(FORMAT("Maximum inlier ratio: {0:.3f}" , MAX(homography_inlier_ratios)))
```

### Algorithm 11 — 10. Visualize Representative Tracking Frames

```text
PROCEDURE read_video_frame_at_index(video_path, target_index)
    capture ← CV.VideoCapture(STRING(video_path))
    IF (NOT capture.isOpened()) THEN
        RAISE RuntimeError(FORMAT("Could not open video: {0}" , video_path))
    END IF
    CALL capture.set(CV.CAP_PROP_POS_FRAMES, INTEGER(target_index))
    (ret, frame) ← capture.read()
    CALL capture.release()
    IF ((NOT ret) OR (frame IS NULL)) THEN
        RAISE RuntimeError(FORMAT("Could not read frame {0}." , target_index))
    END IF
    RETURN frame
END PROCEDURE
n_examples ← MIN(6, LENGTH(successful_tracking_results))
selected_result_indices ← ARRAY.linspace(0, (LENGTH(successful_tracking_results) − 1), n_examples,
    dtype = INTEGER)
selected_tracking_results ← LIST[successful_tracking_results[index] FOR index IN selected_result_indices]
n_cols ← 3
n_rows ← INTEGER(ARRAY.ceil((n_examples / n_cols)))
(fig, axes) ← PLOT.subplots(n_rows, n_cols, figsize = (15, (5 * n_rows)))
axes ← RESHAPE(ARRAY.asarray(axes), (−1))
FOR (ax, result) IN ZIP(axes, selected_tracking_results) DO
    frame ← read_video_frame_at_index(VIDEO_PATH, result["frame_index"])
    CALL CV.polylines(frame, [ARRAY.int32(result["current_bbox"])], TRUE, (0, 255, 0), 3, CV.LINE_AA)
    frame_rgb ← CV.cvtColor(frame, CV.COLOR_BGR2RGB)
    CALL ax.imshow(frame_rgb)
    CALL ax.set_title(FORMAT("Frame {0} | Inliers: {1}/{2}" , result["frame_index"], result["num_inliers"],
        result["num_matches"]))
    CALL ax.axis("off")
END FOR
FOR ax IN axes[n_examples:] DO
    CALL ax.axis("off")
END FOR
CALL PLOT.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "representative_tracking_frames.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
```

### Algorithm 12 — 11. Visualize RANSAC Inlier Matches

```text
example_tracking_result ← successful_tracking_results[(LENGTH(successful_tracking_results) DIV 2)]
example_video_frame ← read_video_frame_at_index(VIDEO_PATH, example_tracking_result["frame_index"])
example_gray_frame ← CV.cvtColor(example_video_frame, CV.COLOR_BGR2GRAY)
example_feature_match_result ← match_features_and_estimate_homography(gray_frame = example_gray_frame,
    reference_keypoints = reference_keypoints, reference_descriptors = reference_descriptors,
    orb_detector = orb, descriptor_matcher = matcher)
IF (NOT example_feature_match_result["success"]) THEN
    RAISE RuntimeError("Representative frame could not be re-matched for visualization.")
END IF
inlier_match_mask ← TO_LIST(CAST(example_feature_match_result["inlier_mask"], ARRAY.uint8))
reference_match_display ← COPY(reference_frame)
CALL CV.polylines(reference_match_display, [ARRAY.int32(initial_bounding_box)], TRUE, (0, 255, 0), 3,
    CV.LINE_AA)
current_match_display ← COPY(example_video_frame)
CALL CV.polylines(current_match_display, [ARRAY.int32(example_tracking_result["current_bbox"])], TRUE, (0,
    255, 0), 3, CV.LINE_AA)
match_visualization ← CV.drawMatches(reference_match_display, reference_keypoints, current_match_display,
    example_feature_match_result["keypoints"], example_feature_match_result["matches"], NULL,
    matchColor = (0, 255, 0), singlePointColor = NULL, matchesMask = inlier_match_mask,
    flags = CV.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
match_visualization_rgb ← CV.cvtColor(match_visualization, CV.COLOR_BGR2RGB)
(fig, ax) ← PLOT.subplots(figsize = (16, 7))
CALL ax.imshow(match_visualization_rgb)
CALL ax.set_title(FORMAT("RANSAC Inlier Matches — Frame {0}" , example_tracking_result["frame_index"]))
CALL ax.axis("off")
CALL PLOT.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "ransac_inlier_matches.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
```

### Algorithm 13 — 12. Analyze Matches, Inliers, and Inlier Ratio Across the Sequence

```text
(fig, ax) ← PLOT.subplots(figsize = (11, 5))
CALL ax.plot(tracked_frame_indices, feature_match_counts, marker = ".", label = "Descriptor matches")
CALL ax.plot(tracked_frame_indices, homography_inlier_counts, marker = ".", label = "RANSAC inliers")
CALL ax.set_title("Feature Matches and RANSAC Inliers by Frame")
CALL ax.set_xlabel("Frame index")
CALL ax.set_ylabel("Count")
CALL ax.grid(alpha = 0.25)
CALL ax.legend()
CALL PLOT.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "matches_and_inliers_by_frame.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
(fig, ax) ← PLOT.subplots(figsize = (11, 5))
CALL ax.plot(tracked_frame_indices, homography_inlier_ratios, marker = ".")
CALL ax.set_title("RANSAC Inlier Ratio by Frame")
CALL ax.set_xlabel("Frame index")
CALL ax.set_ylabel("Inlier ratio")
CALL ax.set_ylim(0, 1.05)
CALL ax.grid(alpha = 0.25)
CALL PLOT.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "inlier_ratio_by_frame.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
```

### Algorithm 14 — 13. Run Numerical and Output-file Validation Checks

```text
IF ((reference_descriptors IS NULL) OR (reference_descriptors.shape[0] < MIN_MATCHES) OR (reference_descriptors.shape[1] ≠ 32)) THEN
    RAISE ValueError("Invalid reference ORB descriptors.")
END IF
IF (LENGTH(frame_tracking_results) = 0) THEN
    RAISE ValueError("No video frames were processed after the reference frame.")
END IF
IF ((reported_frame_count > 0) AND (LENGTH(frame_tracking_results) ≠ (reported_frame_count − 1))) THEN
    RAISE ValueError("Processed-frame count does not match the video frame count.")
END IF
FOR result IN successful_tracking_results DO
    H ← result["homography"]
    tracked_bounding_box ← result["current_bbox"]
    IF ((H.shape ≠ (3, 3)) OR (NOT ARRAY.all(ARRAY.isfinite(H)))) THEN
        RAISE ValueError(FORMAT("Invalid homography in frame {0}." , result["frame_index"]))
    END IF
    IF ((tracked_bounding_box.shape ≠ (4, 1, 2)) OR (NOT ARRAY.all(ARRAY.isfinite(tracked_bounding_box)))) THEN
        RAISE ValueError(FORMAT("Invalid transformed bounding box in frame {0}." , result["frame_index"]))
    END IF
    IF ((result["num_matches"] < MIN_MATCHES) OR (result["num_inliers"] < MIN_INLIERS) OR (result["num_inliers"] > result["num_matches"])) THEN
        RAISE ValueError(FORMAT("Invalid correspondence statistics in frame {0}." , result["frame_index"]))
    END IF
END FOR
IF (NOT ARRAY.all(ARRAY.isfinite(homography_inlier_ratios))) THEN
    RAISE ValueError("Non-finite inlier ratio detected.")
END IF
IF (ARRAY.any((homography_inlier_ratios < 0.0)) OR ARRAY.any((homography_inlier_ratios > 1.0))) THEN
    RAISE ValueError("Inlier ratios outside [0, 1].")
END IF
REQUIRED_OUTPUTS ← ["reference_orb_keypoints.png", "representative_tracking_frames.png", "ransac_inlier_matches.png",
    "matches_and_inliers_by_frame.png", "inlier_ratio_by_frame.png"]
missing_outputs ← LIST[name FOR name IN REQUIRED_OUTPUTS WHERE (NOT (OUTPUT_DIR / name).exists())]
IF missing_outputs THEN
    RAISE FileNotFoundError(("Missing outputs: " + ", ".join(missing_outputs)))
END IF
CALL EMIT("All Feature Detection validation checks passed.")
```

### Completeness check

| Executable element | Included in the pseudocode |
| --- | ---: |
| Source code cells | 14 |
| Procedure bodies | 2 |
| Assignments | 85 |
| Conditional branches | 29 |
| Loops | 4 |
| Assertions | 0 |
| Explicit error branches | 21 |

The counts check transcription coverage, not mathematical correctness or a successful execution. Every original diagnostic and rendering call is included. The limitations discussed below still apply.

## Task Coverage

Every task defined in the Problem Statement is covered by the coding sequence above:

- **Task 1:** Validate the Input Video and Output Paths
- **Task 2:** Define the Initial Bounding Box and Tracking Parameters
- **Task 3:** Initialize ORB and the Hamming-Distance Matcher
- **Task 4:** Read and Validate the Reference Frame
- **Task 5:** Detect Reference ORB Features Inside the Object Region
- **Task 6:** Visualize the Reference Object and ORB Keypoints
- **Task 7:** Define Frame Matching and RANSAC Homography Estimation
- **Task 8:** Track the Object Throughout the Video
- **Task 9:** Compute Tracking Summary Metrics
- **Task 10:** Visualize Representative Tracking Frames
- **Task 11:** Visualize RANSAC Inlier Matches
- **Task 12:** Analyze Matches, Inliers, and Inlier Ratio Across the Sequence
- **Task 13:** Run Numerical and Output-file Validation Checks


## Notebook Relationship

- `problem_statement.ipynb` defines **what must be solved**.
- `theory.ipynb` explains **why the methods work**.
- `requirements.ipynb` defines **what is needed to run the lab**.
- `algorithm.ipynb` defines **how to build the solution step by step**.
- `main.ipynb` is the **executable implementation**.


## Required Output Contract

The current `REQUIRED_OUTPUTS` list in `main.ipynb` contains 5 files under `../outputs/figures/`:

- `reference_orb_keypoints.png`
- `representative_tracking_frames.png`
- `ransac_inlier_matches.png`
- `matches_and_inliers_by_frame.png`
- `inlier_ratio_by_frame.png`

A presence check detects missing artifacts, not stale artifacts. Inspect the figures from the run being reported.

## Implementation Traceability

Reviewed against [source revision 66078c2](https://github.com/denoskume/msc-coro-dassip-portfolio/tree/66078c2481115d0b2596c6926e0750d136ba1005/labs/computer-vision/feature-tracking/notebooks).

The following helpers/classes already exist in `main.ipynb`; algorithm operations that are not listed here are inline notebook cells or descriptive pseudocode:

- `match_features_and_estimate_homography`
- `read_video_frame_at_index`

## Implementation Limits and Interpretation

- Reference descriptors remain fixed to the first frame; this is reference-to-frame homography tracking, not optical flow or a frame-to-frame descriptor update.
- The inlier-match figure reruns matching/RANSAC. It is a diagnostic rerun and may differ from the stored sequence estimate; the sequence metrics still use recorded tracking results.
- Four matches/inliers are necessary minima for this implementation, not sufficient proof of a well-conditioned mapping. Finite boxes and high inlier ratio do not establish localization accuracy without ground truth.
- Report decoding/frame-count limitations separately from recorded tracking failures.

## 15. End

The coding plan ends after the implemented checks and required artifacts have been reviewed. Interpret outputs beside their corresponding experiment, then write the overall analysis at the bottom of `main.ipynb`. Report the implementation limits below; a numerical pass alone is not proof of physical validity or application-level accuracy.